In [5]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [8]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

Using: mps


### Function calculate the perplexity 
Perplexity is the negative log likelihood of loss. In other words, we are calculating the negative log likelihood of predicting a word in the sentence given all the previous words. 
Higher the perplexity less the confidence of the model is in its output.
Perplexity of 1 is the ideal case which shows absolute certainty on its output. 

In [9]:
def calculate_perplexity(model, tokenizer, text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt')

    # Get the model's output logits
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        loss = outputs.loss

    # Calculate perplexity
    perplexity = torch.exp(loss)
    return perplexity.item()

## Model and Tokenizer Initialization

In [10]:
model = GPT2LMHeadModel.from_pretrained('GPT2')
tokenizer = GPT2Tokenizer.from_pretrained('GPT2')

## Testing Perplexity Score Calculation

In [11]:
clean_text = "The quick brown fox jumps over the lazy dog."
noisy_text = "Th3 qu1ck br0wn f0x jumps 0ver th3 l@zy d0g." 

In [ ]:
clean_perplexity = calculate_perplexity(model, tokenizer, clean_text)
noisy_perplexity = calculate_perplexity(model, tokenizer, noisy_text)

print(f"Perplexity of clean text: {clean_perplexity:.2f}")
print(f"Perplexity of noisy text: {noisy_perplexity:.2f}")

### Fixing data quality issues

In [25]:
%pip install pyspellchecker
import spacy
from collections import Counter
from spellchecker import SpellChecker

spell = SpellChecker()

     |████████████████████████████████| 7.2 MB 216 kB/s eta 0:00:01
You should consider upgrading via the '/Users/saminsadaf/Documents/LLM-Design-Patterns/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [26]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")
def analyze_text_quality(text):
    doc = nlp(text)
    
    # Check spelling with pyspellchecker; spaCy has no built-in misspelling extension.
    words = [token.text.lower() for token in doc if token.is_alpha]
    misspelled = spell.unknown(words)
    misspelled_words = [
        token.text for token in doc
        if token.is_alpha and token.text.lower() in misspelled
    ]
    
    pos_counts = Counter(token.pos_ for token in doc)
    grammar_score = pos_counts['NOUN'] + pos_counts['VERB'] + pos_counts['ADJ'] + pos_counts['ADV']
    
    incomplete_sentences = [
        sent.text for sent in doc.sents if len(sent) < 3
    ]
    
    return {
        "misspelled_words": misspelled_words,
        "grammar_score": grammar_score,
        "incomplete_sentences": incomplete_sentences
    }

In [27]:
text = "This iz a smple txt with sum issues. Incomlet"
quality_report = analyze_text_quality(text)
print(quality_report)

{'misspelled_words': ['iz', 'smple', 'txt', 'Incomlet'], 'grammar_score': 5, 'incomplete_sentences': ['Incomlet']}


### Text Preprocessing 

In [34]:
import unicodedata
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

In [35]:
#Download required NLTK Data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/saminsadaf/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/saminsadaf/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/saminsadaf/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [36]:
def preprocess_text(text):
    #Lowercase the text
    text = text.lower()
    # Normalize the unicode characters
    text = unicodedata.normalize(
        'NFKD', text
    ).encode(
        'ascii', 'ignore'
    ).decode('utf-8')
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    #Normalize the whitespace
    text = ' '.join(text.split())
    # Tokenize the text
    tokens = word_tokenize(text)
    
    #Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [
        token for token in tokens if token not in stop_words
    ]
    
    #Join tokens back into text
    
    preprocessed_text = ' '.join(tokens)
    
    return preprocessed_text

In [37]:
raw_text = "This is an EXAMPLE of text preprocessing... It's quite useful!"
cleaned_text = preprocess_text(raw_text)
print(f"Original Text: {raw_text}")
print(f"Cleaned Text: {cleaned_text}")

Original Text: This is an EXAMPLE of text preprocessing... It's quite useful!
Cleaned Text: example text preprocessing quite useful
